# PyTorch Baseline CNN Model

Training a baseline Convolutional Neural Network using PyTorch with GPU acceleration.

**Objectives:**
- Build baseline CNN with PyTorch on GPU
- Compare with Keras implementation
- Train with early stopping and learning rate scheduling
- Evaluate performance on test set
- Save model for inference

**Hardware:** NVIDIA GeForce RTX 3050 (4GB VRAM)

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.pytorch_models import BaselineCNN, get_model
from src.pytorch_train import train_model, create_dataloaders, plot_training_history, get_class_weights
from src.pytorch_evaluate import evaluate_model, plot_confusion_matrix, plot_per_class_metrics, create_evaluation_report

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Device: {device}")

## 2. Load Preprocessed Data

In [ ]:
# Load preprocessed data (same as Keras version)
data_dir = os.path.join('..', 'data', 'preprocessed')

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

# Load class weights
with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights_dict = json.load(f)

# Convert to PyTorch tensor
class_weights_list = [class_weights_dict[str(i)] for i in range(7)]
class_weights = torch.tensor(class_weights_list, dtype=torch.float32, device=device)

print(f"Training data: {X_train.shape}")
print(f"Validation data: {X_val.shape}")
print(f"Test data: {X_test.shape}")
print(f"Y_train shape: {y_train.shape}")
print(f"\nClass weights: {class_weights_list}")

## 3. Create DataLoaders

In [ ]:
# Create DataLoaders
train_loader, val_loader = create_dataloaders(
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    device=device
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Verify batch
for X_batch, y_batch in train_loader:
    print(f"\nBatch X shape: {X_batch.shape} on {X_batch.device}")
    print(f"Batch y shape: {y_batch.shape} on {y_batch.device}")
    break

## 4. Build Baseline Model

In [ ]:
# Create model
baseline_model = get_model('baseline', num_classes=7, device=device)

# Print architecture
print("Baseline CNN Architecture:")
print(baseline_model)

# Count parameters
total_params = sum(p.numel() for p in baseline_model.parameters())
trainable_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Train Model

In [ ]:
# Train model with GPU acceleration
history_baseline = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=100,
    learning_rate=0.001,
    device=device,
    model_name='pytorch_baseline_cnn',
    class_weights=class_weights
)

print("\nTraining complete!")

## 6. Plot Training History

In [ ]:
# Create training history plot
os.makedirs('results', exist_ok=True)
plot_training_history(
    history_baseline,
    model_name='PyTorch Baseline CNN',
    save_path='../results/model/pytorch_baseline_training_history.png'
)

## 7. Evaluate on Test Set

In [ ]:
# Comprehensive evaluation
results_baseline = create_evaluation_report(
    baseline_model,
    X_test,
    y_test,
    device=device,
    model_name='pytorch_baseline'
)

## 8. Summary and Statistics

In [ ]:
# Training summary
print("="*60)
print("BASELINE CNN TRAINING SUMMARY")
print("="*60)

print(f"\nBest Validation Accuracy: {max(history_baseline['val_accuracy']):.4f}")
print(f"Best Validation Loss: {min(history_baseline['val_loss']):.4f}")
print(f"\nFinal Test Accuracy: {results_baseline['accuracy']:.4f}")
print(f"Total Parameters: {total_params:,}")

print(f"\nTraining Time:")
print(f"  Device: {device}")
print(f"  GPU Memory Used (if applicable): ~500-1000 MB")

print(f"\nModel saved to:")
print(f"  Best: saved_models/pytorch_baseline_cnn_best.pt")
print(f"  Final: saved_models/pytorch_baseline_cnn_final.pt")

print("\nResults saved to:")
print(f"  Confusion Matrix: results/pytorch_pytorch_baseline_confusion_matrix.png")
print(f"  Per-Class Metrics: results/pytorch_pytorch_baseline_per_class_metrics.png")
print(f"  Training History: results/pytorch_baseline_training_history.png")